# Lab prob_01 — Monte Carlo: LLN, CLT, birthdays and MLE

**Learning goals**
1. Estimate probabilities by simulation and see the error shrink like $1/\sqrt{n}$.
2. Watch the **law of large numbers** and the **central limit theorem** happen.
3. Check the birthday problem: exact formula vs Monte Carlo.
4. Code maximum-likelihood estimators (Bernoulli, Poisson) and a **Beta-Binomial** posterior.

Do the paper homework for *Expectation, variance, linearity* first — this lab assumes it.

**Reading:** [Harvard Stat 110](https://stat110.hsites.harvard.edu/) ·
[Blitzstein & Hwang, *Introduction to Probability* (free)](https://probabilitybook.net/) — Ch. 1 (birthday), Ch. 10 (LLN/CLT) ·
[MIT 18.05](https://ocw.mit.edu/courses/18-05-introduction-to-probability-and-statistics-spring-2022/) (MLE, Bayesian updating).

Fill in every cell marked `# TODO`, run the exploration cells, then run the last cell to grade.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(0)

## 1. Monte Carlo estimates

If $I_1,\dots,I_n$ are i.i.d. indicators of an event $A$, then $\hat p_n = \frac1n\sum I_j$ has
$E[\hat p_n] = P(A)$ and $\mathrm{SD}(\hat p_n) = \sqrt{p(1-p)/n}$: to halve the error you need **4×** the samples.

Implement `mc_prob(trial, n, rng)`: `trial(rng)` runs one random experiment and returns `True`/`False`;
return the fraction of `True` over `n` runs.

In [ ]:
def mc_prob(trial, n, rng):
    """Monte-Carlo estimate of P(event) from n calls of trial(rng)."""
    # TODO
    raise NotImplementedError

In [ ]:
# Exploration: P(sum of two dice = 7) and the 1/sqrt(n) error
two_dice_7 = lambda r: r.integers(1, 7) + r.integers(1, 7) == 7
for n in [100, 1_000, 10_000, 100_000]:
    est = mc_prob(two_dice_7, n, rng)
    print(f"n={n:>7}: estimate {est:.4f}   error {abs(est - 1/6):.4f}   theory SD {math.sqrt((1/6)*(5/6)/n):.4f}")

## 2. Law of large numbers and the CLT

**LLN:** $\bar X_n \to \mu$. **CLT:** $\dfrac{\bar X_n - \mu}{\sigma/\sqrt n} \approx \mathcal N(0,1)$ for large $n$,
*whatever* the distribution of $X$ (finite variance).

Implement
* `sample_means(sampler, n, reps, rng)` — `sampler(rng, size)` returns i.i.d. draws with the given numpy `size`
  (an int or a tuple); return a 1-D array of `reps` sample means, each over `n` draws;
* `standardize_means(means, mu, sigma, n)` — the CLT z-scores.

In [ ]:
def sample_means(sampler, n, reps, rng):
    """Array of `reps` sample means, each of n draws from sampler(rng, size)."""
    # TODO (hint: draw a (reps, n) array in one call and average along axis 1)
    raise NotImplementedError


def standardize_means(means, mu, sigma, n):
    """(mean - mu) / (sigma / sqrt(n))"""
    # TODO
    raise NotImplementedError

In [ ]:
# Exploration: running mean of fair-coin flips (LLN)
flips = rng.integers(0, 2, 5000)
plt.plot(np.cumsum(flips) / np.arange(1, 5001))
plt.axhline(0.5, color="k", lw=0.8)
plt.xscale("log"); plt.xlabel("n"); plt.ylabel("running mean"); plt.title("LLN"); plt.show()

# Exploration: CLT for a skewed Expo(1/2) distribution (mean 2, sd 2)
expo = lambda r, size: r.exponential(2.0, size)
xs = np.linspace(-4, 4, 200)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, n in zip(axes, [1, 5, 50]):
    z = standardize_means(sample_means(expo, n, 20_000, rng), 2.0, 2.0, n)
    ax.hist(z, bins=60, density=True, alpha=0.6)
    ax.plot(xs, np.exp(-xs**2 / 2) / math.sqrt(2 * math.pi), "k")
    ax.set_title(f"n = {n}")
plt.show()

## 3. The birthday problem

With $k$ people and $d$ equally likely days,
$$P(\text{some shared birthday}) = 1 - \prod_{j=0}^{k-1}\frac{d-j}{d},$$
which is $1$ when $k > d$ (pigeonhole). Implement `birthday_exact(k, days=365)` and a **vectorised**
`birthday_mc(k, n_trials, rng, days=365)` (draw an `(n_trials, k)` integer array; a trial "succeeds" if any value repeats —
sorting each row and checking `np.diff == 0` is one way).

In [ ]:
def birthday_exact(k, days=365):
    # TODO
    raise NotImplementedError


def birthday_mc(k, n_trials, rng, days=365):
    # TODO
    raise NotImplementedError

In [ ]:
ks = np.arange(1, 61)
plt.plot(ks, [birthday_exact(k) for k in ks], label="exact")
plt.plot(ks[::4], [birthday_mc(k, 4000, rng) for k in ks[::4]], "o", label="Monte Carlo")
plt.axhline(0.5, color="k", lw=0.8); plt.xlabel("k people"); plt.legend(); plt.show()
print("smallest k with P >= 1/2:", next(k for k in ks if birthday_exact(k) >= 0.5))

## 4. Maximum likelihood

For i.i.d. $\mathrm{Bern}(p)$: $\ell(p) = k\log p + (n-k)\log(1-p)$, so $\hat p = \bar x$.
For i.i.d. $\mathrm{Pois}(\lambda)$:
$$\ell(\lambda) = \sum_i \big[x_i\log\lambda - \lambda - \log x_i!\big],\qquad \ell'(\lambda) = \frac{\sum x_i}{\lambda} - n = 0 \Rightarrow \hat\lambda = \bar x.$$
Implement `bernoulli_mle(x)`, `poisson_mle(x)` and `poisson_loglik(lam, x)` (include the $-\log x_i!$ term; `math.lgamma(v + 1)` $=\log v!$).

In [ ]:
def bernoulli_mle(x):
    # TODO
    raise NotImplementedError


def poisson_mle(x):
    # TODO
    raise NotImplementedError


def poisson_loglik(lam, x):
    # TODO
    raise NotImplementedError

In [ ]:
data = rng.poisson(3.2, 40)
grid = np.linspace(1.5, 5, 300)
plt.plot(grid, [poisson_loglik(l, data) for l in grid])
plt.axvline(poisson_mle(data), color="r", ls="--", label=f"MLE = {poisson_mle(data):.3f}")
plt.xlabel("lambda"); plt.ylabel("log-likelihood"); plt.legend(); plt.show()

## 5. Beta-Binomial posterior

Prior $p\sim\mathrm{Beta}(a,b)$, data $k$ successes in $n$ trials $\Rightarrow$ posterior $\mathrm{Beta}(a+k,\ b+n-k)$,
with mean $\frac{a+k}{a+b+n}$ — a compromise between the prior mean $\frac{a}{a+b}$ and the MLE $\frac kn$.
Implement `beta_binomial_posterior(a, b, k, n)` (return a tuple) and `beta_mean(a, b)`.

In [ ]:
def beta_binomial_posterior(a, b, k, n):
    # TODO
    raise NotImplementedError


def beta_mean(a, b):
    # TODO
    raise NotImplementedError

In [ ]:
def beta_pdf(p, a, b):
    return np.exp((a - 1) * np.log(p) + (b - 1) * np.log1p(-p) - (math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)))

ps = np.linspace(0.001, 0.999, 400)
a, b = 2, 2
plt.plot(ps, beta_pdf(ps, a, b), "k--", label="prior Beta(2,2)")
for n in [5, 20, 100]:
    k = rng.binomial(n, 0.7)
    a_post, b_post = beta_binomial_posterior(a, b, k, n)
    plt.plot(ps, beta_pdf(ps, a_post, b_post), label=f"n={n}, k={k}: mean {beta_mean(a_post, b_post):.3f}")
plt.axvline(0.7, color="gray", lw=0.8); plt.legend(); plt.title("Posterior concentrates around the truth (0.7)"); plt.show()

## Grade
Runs the hidden checks (fresh inputs, fixed seeds) and records the result in the Learning Center.

In [ ]:
from lc_client import grade
grade("prob_01_monte_carlo", globals())